# 实践实验 - 应用机器学习的建议
在本实验中，你将探索评估和改进机器学习模型的技术。

# 目录
- [ 1 - 程序包 ](#1)
- [ 2 - 评估学习算法（多项式回归）](#2)
  - [ 2.1 划分数据集](#2.1)
  - [ 2.2 模型评估的误差计算，线性回归](#2.2)
    - [ 练习 1](#ex01)
  - [ 2.3 比较训练数据与测试数据上的表现](#2.3)
- [ 3 - 偏差与方差<img align="Right" src="./图片/C2_W3_BiasVarianceDegree.png"  style=" width:500px; padding: 10px 20px ; ">

> **图中文字（中文）**：诊断偏差与方差 如何判断你的算法存在偏差问题还是方差问题？ 高偏差（欠拟合）：J_train 会很高 高方差（过拟合）：J_cv ≫ J_train，J_train 可能很低 （图：J_train 与 J_cv 随多项式次数（degree of polynomial）的变化） ](#3)
  - [ 3.1 绘制训练集、交叉验证集、测试集](#3.1)
  - [ 3.2 寻找最优次数](#3.2)
  - [ 3.3 调整正则化](#3.3)
  - [ 3.4 获取更多数据：增大训练集规模 (m)](#3.4)
- [ 4 - 评估学习算法（神经网络）](#4)
  - [ 4.1 数据集](#4.1)
  - [ 4.2 通过计算分类误差来评估分类模型](#4.2)
    - [ 练习 2](#ex02)
- [ 5 - 模型复杂度](#5)
  - [ 练习 3](#ex03)
  - [ 5.1 简单模型](#5.1)
    - [ 练习 4](#ex04)
- [ 6 - 正则化](#6)
  - [ 练习 5](#ex05)
- [ 7 - 迭代寻找最优正则化取值](#7)
  - [ 7.1 测试](#7.1)


<a name="1"></a>
## 1 - 程序包

首先，让我们运行下面的单元格，导入本作业需要用到的所有程序包。
- [numpy](https://numpy.org/) 是 Python 科学计算的基础程序包。
- [matplotlib](http://matplotlib.org) 是 Python 中常用的绘图库。
- [scikitlearn](https://scikit-learn.org/stable/) 是数据挖掘的基础库
- [tensorflow](https://www.tensorflow.org/) 是流行的机器学习平台。

In [ ]:
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import relu,linear
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.optimizers import Adam

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

from public_tests_a1 import * 

tf.keras.backend.set_floatx('float64')
from assigment_utils import *

tf.autograph.set_verbosity(0)

<a name="2"></a>
## 2 - 评估学习算法（多项式回归）

<img align="Right" src="./图片/C2_W3_TrainingVsNew.png"  style=" width:350px; padding: 10px 20px ; ">

> **图中文字（中文）**：训练数据 vs 新数据 模型：w1·x1 + w2·x2 + … + wn·xn + b （横轴：尺寸 size） 假设你创建了一个机器学习模型，发现它非常*好地拟合*了训练数据。这样就完事了吗？并不尽然。创建模型的目标是能够对<span style="color:blue">*新*</span>样本预测取值。

在部署之前，你如何测试模型在新数据上的表现？  
答案包含两部分：
* 把原始数据集划分为“训练”集和“测试”集。
    * 用训练数据拟合模型参数
    * 用测试数据在*新*数据上评估模型
* 设计一个误差函数来评估模型。

<a name="2.1"></a>
### 2.1 划分数据集
课程建议留出数据集的 20-40% 用于测试。让我们用 `sklearn` 的函数 [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) 来执行划分。运行下面的单元格后请再核对一下形状。

In [ ]:
# 生成一些数据
X,y,x_ideal,y_ideal = gen_data(18, 2, 0.7)
print("X.shape", X.shape, "y.shape", y.shape)

#用 sklearn 程序划分数据
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33, random_state=1)
print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_test.shape", X_test.shape, "y_test.shape", y_test.shape)

#### 2.1.1 绘制训练集、测试集
下面可以看到，将用于训练的数据点（红色）与模型未训练过的数据点（测试集）混杂在一起。这个特定数据集是一个加了噪声的二次函数。“理想”曲线也画了出来供参考。

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(x_ideal, y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
ax.set_title("Training, Test",fontsize = 14)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(X_train, y_train, color = "red",           label="train")
ax.scatter(X_test, y_test,   color = dlc["dlblue"],   label="test")
ax.legend(loc='upper left')
plt.show()

<a name="2.2"></a>
### 2.2 模型评估的误差计算，线性回归
在*评估*线性回归模型时，你对预测值与目标值之差的平方取平均。

$$ J_\text{test}(\mathbf{w},b) = 
            \frac{1}{2m_\text{test}}\sum_{i=0}^{m_\text{test}-1} ( f_{\mathbf{w},b}(\mathbf{x}^{(i)}_\text{test}) - y^{(i)}_\text{test} )^2 
            \tag{1}
$$

<a name="ex01"></a>
### 练习 1

下面，为线性回归模型创建一个在数据集上评估误差的函数。

In [ ]:
# UNQ_C1
# 计分单元：eval_mse
def eval_mse(y, yhat):
    """在数据集上计算均方误差。
    参数：
      y    : (ndarray  形状 (m,) 或 (m,1))  每个样本的目标值
      yhat : (ndarray  形状 (m,) 或 (m,1))  每个样本的预测值
    返回：
      err: (scalar)"""
    m = len(y)
    err = 0.0
    for i in range(m):
    ### 在此处编写代码 ###
    
    ### 代码编写结束 ###
    
    return(err)

In [ ]:
y_hat = np.array([2.4, 4.2])
y_tmp = np.array([2.3, 4.1])
eval_mse(y_hat, y_tmp)

# 单元测试开始
test_eval_mse(eval_mse)   
# 单元测试结束

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>

    

```python
def eval_mse(y, yhat):
    """ 
    Calculate the mean squared error on a data set.
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:
      err: (scalar)             
    """
    m = len(y)
    err = 0.0
    for i in range(m):
        err_i  = ( (yhat[i] - y[i])**2 ) 
        err   += err_i                                                                
    err = err / (2*m)                    
    return(err)
```
 

<a name="2.3"></a>
### 2.3 比较训练数据与测试数据上的表现
让我们构建一个高次多项式模型来最小化训练误差。这里会用到 `sklearn` 的 linear_regression 函数。如果你想看细节，代码在导入的工具文件中。步骤如下：
* 创建并拟合模型。（'fit' 是训练或运行梯度下降的另一种说法。）
* 计算训练数据上的误差。
* 计算测试数据上的误差。

In [ ]:
# 在 sklearn 中创建模型，并在训练数据上训练
degree = 10
lmodel = lin_model(degree)
lmodel.fit(X_train, y_train)

# 在训练数据上预测，求训练误差
yhat = lmodel.predict(X_train)
err_train = lmodel.mse(y_train, yhat)

# 在测试数据上预测，求误差
yhat = lmodel.predict(X_test)
err_test = lmodel.mse(y_test, yhat)

训练集上算出的误差远小于测试集上的误差。

In [ ]:
print(f"training err {err_train:0.2f}, test err {err_test:0.2f}")

下面的图说明了原因。模型非常拟合训练数据。为做到这一点，它构造了一个复杂的函数。测试数据不属于训练的一部分，模型在这份数据上预测得很差。  
这个模型可以被描述为：1）过拟合，2）高方差，3）“泛化”能力差。

In [ ]:
# 在数据范围内绘制预测曲线
x = np.linspace(0,int(X.max()),100)  # 为绘图预测取值
y_pred = lmodel.predict(x).reshape(-1,1)

plt_train_test(X_train, y_train, X_test, y_test, x, y_pred, x_ideal, y_ideal, degree)

测试集误差表明这个模型在新数据上效果不佳。如果你用测试误差来指导模型改进，模型在测试数据上会表现良好……但测试数据的本意是代表*新*数据。
你需要再另有一份数据来检验新数据上的表现。

课程中提出的方案是把数据分成三组。下表中训练集、交叉验证集和测试集的分布是一种典型分布，但可以根据可用数据量做调整。

| 数据             | 占总量的比例 | 说明 |
|------------------|:----------:|:---------|
| 训练集           | 60         | 用于在训练或拟合中调整模型参数 $w$ 和 $b$ 的数据 |
| 交叉验证集       | 20         | 用于调整其他模型参数（如多项式次数、正则化，或神经网络架构）的数据。|
| 测试集           | 20         | 在调参之后用于检验模型在新数据上表现的数据 |


让我们在下面生成三个数据集。我们将再次使用 `sklearn` 的 `train_test_split`，但调用两次以得到三个划分：

In [ ]:
# 生成数据
X,y, x_ideal,y_ideal = gen_data(40, 5, 0.7)
print("X.shape", X.shape, "y.shape", y.shape)

#用 sklearn 程序划分数据
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.40, random_state=1)
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.50, random_state=1)
print("X_train.shape", X_train.shape, "y_train.shape", y_train.shape)
print("X_cv.shape", X_cv.shape, "y_cv.shape", y_cv.shape)
print("X_test.shape", X_test.shape, "y_test.shape", y_test.shape)

<a name="3"></a>
## 3 - 偏差与方差<img align="Right" src="./图片/C2_W3_BiasVarianceDegree.png"  style=" width:500px; padding: 10px 20px ; ">

> **图中文字（中文）**：诊断偏差与方差 如何判断你的算法存在偏差问题还是方差问题？ 高偏差（欠拟合）：J_train 会很高 高方差（过拟合）：J_cv ≫ J_train，J_train 可能很低 （图：J_train 与 J_cv 随多项式次数（degree of polynomial）的变化） 
上面已经清楚，多项式模型的次数太高了。你该如何选择一个好的取值？结果表明，如图中所示，训练表现和交叉验证表现可以提供指导。通过尝试一系列次数取值，可以评估训练表现和交叉验证表现。当次数变得过大时，交叉验证表现相对于训练表现会开始变差。让我们在自己的例子上试一试。

<a name="3.1"></a>
### 3.1 绘制训练集、交叉验证集、测试集
下面可以看到，将用于训练的数据点（红色）与模型未训练过的数据点（测试集和交叉验证集）混杂在一起。

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(4,4))
ax.plot(x_ideal, y_ideal, "--", color = "orangered", label="y_ideal", lw=1)
ax.set_title("Training, CV, Test",fontsize = 14)
ax.set_xlabel("x")
ax.set_ylabel("y")

ax.scatter(X_train, y_train, color = "red",           label="train")
ax.scatter(X_cv, y_cv,       color = dlc["dlorange"], label="cv")
ax.scatter(X_test, y_test,   color = dlc["dlblue"],   label="test")
ax.legend(loc='upper left')
plt.show()

<a name="3.2"></a>
### 3.2 寻找最优次数
在之前的实验中，你发现可以利用多项式构建出能拟合复杂曲线的模型（见课程 1 第 2 周的特征工程与多项式回归实验）。此外你还演示了：通过增加多项式的*次数*，可以*制造*过拟合。（见课程 1 第 3 周的过拟合实验）。让我们在这里运用这些知识，检验我们区分过拟合与欠拟合的能力。

让我们反复训练模型，每次迭代都增加多项式的次数。这里，为了速度和简洁，我们使用 [scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression) 的线性回归模型。

In [ ]:
max_degree = 9
err_train = np.zeros(max_degree)    
err_cv = np.zeros(max_degree)      
x = np.linspace(0,int(X.max()),100)  
y_pred = np.zeros((100,max_degree))  #各列是要绘制的折线

for degree in range(max_degree):
    lmodel = lin_model(degree+1)
    lmodel.fit(X_train, y_train)
    yhat = lmodel.predict(X_train)
    err_train[degree] = lmodel.mse(y_train, yhat)
    yhat = lmodel.predict(X_cv)
    err_cv[degree] = lmodel.mse(y_cv, yhat)
    y_pred[:,degree] = lmodel.predict(x)
    
optimal_degree = np.argmin(err_cv)+1

<font size="4">让我们画出结果：</font>

In [ ]:
plt.close("all")
plt_optimal_degree(X_train, y_train, X_cv, y_cv, x, y_pred, x_ideal, y_ideal, 
                   err_train, err_cv, optimal_degree, max_degree)

上图表明，把数据分成两组——模型训练过的数据和模型未训练过的数据——可以用来判断模型是欠拟合还是过拟合。在我们的例子中，通过增加所用多项式的次数，我们造出了从欠拟合到过拟合的各种模型。
- 左图中，实线表示这些模型的预测。1 次多项式模型产生一条只穿过极少数据点的直线，而最高次数则几乎紧贴每一个数据点。
- 右图中：
    - 如预期那样，训练数据上的误差（蓝色）随着模型复杂度增加而下降
    - 交叉验证数据上的误差起初随着模型开始贴合数据而下降，但随后随着模型开始对训练数据过拟合（*泛化*失败）而上升。    
    
值得注意的是，这些例子中的曲线不像课上画的那样平滑。很明显，分到各组的特定数据点会显著改变你的结果。重要的是总体趋势。

<a name="3.3"></a>
### 3.3 调整正则化
在之前的实验中，你使用过*正则化*来减少过拟合。与次数类似，也可以用同样的方法来调整正则化参数 lambda（$\lambda$）。

让我们从一个高次多项式出发，改变正则化参数来演示这一点。

In [ ]:
lambda_range = np.array([0.0, 1e-6, 1e-5, 1e-4,1e-3,1e-2, 1e-1,1,10,100])
num_steps = len(lambda_range)
degree = 10
err_train = np.zeros(num_steps)    
err_cv = np.zeros(num_steps)       
x = np.linspace(0,int(X.max()),100) 
y_pred = np.zeros((100,num_steps))  #各列是要绘制的折线

for i in range(num_steps):
    lambda_= lambda_range[i]
    lmodel = lin_model(degree, regularization=True, lambda_=lambda_)
    lmodel.fit(X_train, y_train)
    yhat = lmodel.predict(X_train)
    err_train[i] = lmodel.mse(y_train, yhat)
    yhat = lmodel.predict(X_cv)
    err_cv[i] = lmodel.mse(y_cv, yhat)
    y_pred[:,i] = lmodel.predict(x)
    
optimal_reg_idx = np.argmin(err_cv) 

In [ ]:
plt.close("all")
plt_tune_regularization(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, optimal_reg_idx, lambda_range)

上面这些图表明，随着正则化增强，模型从高方差（过拟合）模型转向高偏差（欠拟合）模型。右图中的竖线表示 lambda 的最优取值。在这个例子中，多项式次数设为 10。

<a name="3.4"></a>
### 3.4 获取更多数据：增大训练集规模 (m)
当模型过拟合（高方差）时，收集更多数据可以改善表现。让我们在这里试一试。

In [ ]:
X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, m_range,degree = tune_m()
plt_tune_m(X_train, y_train, X_cv, y_cv, x, y_pred, err_train, err_cv, m_range, degree)

上面这些图表明，当模型高方差且过拟合时，增加样本可以改善表现。注意左图中的曲线。$m$ 取值最大的那条最终曲线是一条位于数据中央的平滑曲线。右图中，随着样本数增加，训练集和交叉验证集的表现收敛到相近的数值。注意这些曲线不像课上看到的那样平滑，这是正常的。趋势依然清晰：更多数据能改善泛化能力。

> 注意，当模型高偏差（欠拟合）时，增加样本并不能改善表现。


<a name="4"></a>
## 4 - 评估学习算法（神经网络）
上面，你调整了多项式回归模型的若干方面。这里，你将使用一个神经网络模型。让我们先创建一个分类数据集。

<a name="4.1"></a>
### 4.1 数据集
运行下面的单元格，生成一个数据集并把它划分为训练集、交叉验证集（CV）和测试集。在这个例子中，为强调效果，我们提高了交叉验证数据点的比例。  

In [ ]:
# 生成并划分数据集
X, y, centers, classes, std = gen_blobs()

# 划分数据。为便于演示使用较大的交叉验证集
X_train, X_, y_train, y_ = train_test_split(X,y,test_size=0.50, random_state=1)
X_cv, X_test, y_cv, y_test = train_test_split(X_,y_,test_size=0.20, random_state=1)
print("X_train.shape:", X_train.shape, "X_cv.shape:", X_cv.shape, "X_test.shape:", X_test.shape)

In [ ]:
plt_train_eq_dist(X_train, y_train,classes, X_cv, y_cv, centers, std)

上面，你可以在左边看到数据。共有六个由颜色区分的簇。训练点（圆点）和交叉验证点（三角形）都显示了出来。有意思的是那些落在模糊位置的点——两个簇都可能认为它们属于自己。你预期神经网络模型会怎么做？什么情况算过拟合？什么算欠拟合？  
右边是一个“理想”模型的例子，即知道数据来源后可能构建出的模型。这些直线表示“等距”边界，即到各中心点距离相等的边界。值得注意的是，这个模型会把大约 8% 的数据“误分类”。

<a name="4.2"></a>
### 4.2 通过计算分类误差来评估分类模型
这里用于分类模型的评估函数就是预测错误的比例：  
$$ J_{cv} =\frac{1}{m}\sum_{i=0}^{m-1} 
\begin{cases}
    1, & \text{if $\hat{y}^{(i)} \neq y^{(i)}$}\\
    0, & \text{otherwise}
\end{cases}
$$

<a name="ex02"></a>
### 练习 2

下面，完成计算分类误差的程序。注意，在本实验中，目标值是类别的索引，而不是[独热编码](https://en.wikipedia.org/wiki/One-hot)。

In [ ]:
# UNQ_C2
# 计分单元：eval_cat_err
def eval_cat_err(y, yhat):
    """计算分类误差
    参数：
      y    : (ndarray  形状 (m,) 或 (m,1))  每个样本的目标值
      yhat : (ndarray  形状 (m,) 或 (m,1))  每个样本的预测值
    返回：|
      cerr: (scalar)"""
    m = len(y)
    incorrect = 0
    for i in range(m):
    ### 在此处编写代码 ###
        
    ### 代码编写结束 ###
    
    return(cerr)

In [ ]:
y_hat = np.array([1, 2, 0])
y_tmp = np.array([1, 2, 3])
print(f"categorization error {np.squeeze(eval_cat_err(y_hat, y_tmp)):0.3f}, expected:0.333" )
y_hat = np.array([[1], [2], [0], [3]])
y_tmp = np.array([[1], [2], [1], [3]])
print(f"categorization error {np.squeeze(eval_cat_err(y_hat, y_tmp)):0.3f}, expected:0.250" )

# 单元测试开始
test_eval_cat_err(eval_cat_err)
# 单元测试结束
# 单元测试开始
test_eval_cat_err(eval_cat_err)
# 单元测试结束

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    

```python
def eval_cat_err(y, yhat):
    """ 
    Calculate the categorization error
    Args:
      y    : (ndarray  Shape (m,) or (m,1))  target value of each example
      yhat : (ndarray  Shape (m,) or (m,1))  predicted value of each example
    Returns:|
      cerr: (scalar)             
    """
    m = len(y)
    incorrect = 0
    for i in range(m):
        if yhat[i] != y[i]:    # @REPLACE
            incorrect += 1     # @REPLACE
    cerr = incorrect/m         # @REPLACE
    return(cerr)                                    
```
 

<a name="5"></a>
## 5 - 模型复杂度
下面，你将构建两个模型：一个复杂模型和一个简单模型。你将评估这些模型，判断它们更可能过拟合还是欠拟合。

###  5.1 复杂模型

<a name="ex03"></a>
### 练习 3
下面，构建一个三层模型：
* 120 个单元的 Dense 层，relu 激活函数
* 40 个单元的 Dense 层，relu 激活函数
* 6 个单元的 Dense 层，线性激活函数（不是 softmax）  
编译时使用
* 配合 `SparseCategoricalCrossentropy` 的损失，记得使用 `from_logits=True`
* 学习率为 0.01 的 Adam 优化器。

In [ ]:
# UNQ_C3
# 计分单元：model
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

tf.random.set_seed(1234)
model = Sequential(
    [
        ### 在此处编写代码 ###
  
        ### 代码编写结束 ###

    ], name="Complex"
)
model.compile(
    ### 在此处编写代码 ###
    loss=None,
    optimizer=None,
    ### 代码编写结束 ###
)

In [ ]:
# 单元测试开始
model.fit(
    X_train, y_train,
    epochs=1000
)
# 单元测试结束

In [ ]:
# 单元测试开始
model.summary()

model_test(model, classes, X_train.shape[1]) 
# 单元测试结束

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
摘要应与下面一致（层实例名称可能递增）

```
Model: "Complex"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 120)               360       
_________________________________________________________________
L2 (Dense)                   (None, 40)                4840      
_________________________________________________________________
L3 (Dense)                   (None, 6)                 246       
=================================================================
Total params: 5,446
Trainable params: 5,446
Non-trainable params: 0
_________________________________________________________________
```

  <details>
  <summary><font size="3" color="darkgreen"><b>点击查看更多提示</b></font></summary>
  

```python
tf.random.set_seed(1234)
model = Sequential(
    [
        Dense(120, activation = 'relu', name = "L1"),      
        Dense(40, activation = 'relu', name = "L2"),         
        Dense(classes, activation = 'linear', name = "L3")  
    ], name="Complex"
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),          
    optimizer=tf.keras.optimizers.Adam(0.01),   
)

model.fit(
    X_train,y_train,
    epochs=1000
)                                  
```
 

In [ ]:
#构建一个模型供绘图程序调用
model_predict = lambda Xl: np.argmax(tf.nn.softmax(model.predict(Xl)).numpy(),axis=1)
plt_nn(model_predict,X_train,y_train, classes, X_cv, y_cv, suptitle="Complex Model")

这个模型费了很大力气去捕捉每个类别的离群点。结果，它把一些交叉验证数据也分错了。让我们计算分类误差。

In [ ]:
training_cerr_complex = eval_cat_err(y_train, model_predict(X_train))
cv_cerr_complex = eval_cat_err(y_cv, model_predict(X_cv))
print(f"categorization error, training, complex model: {training_cerr_complex:0.3f}")
print(f"categorization error, cv,       complex model: {cv_cerr_complex:0.3f}")

<a name="5.1"></a>
### 5.1 简单模型
现在，让我们试一个简单模型

<a name="ex04"></a>
### 练习 4

下面，构建一个两层模型：
* 6 个单元的 Dense 层，relu 激活函数
* 6 个单元的 Dense 层，线性激活函数。  
编译时使用
* 配合 `SparseCategoricalCrossentropy` 的损失，记得使用 `from_logits=True`
* 学习率为 0.01 的 Adam 优化器。

In [ ]:
# UNQ_C4
# 计分单元：model_s

tf.random.set_seed(1234)
model_s = Sequential(
    [
        ### 在此处编写代码 ###
      
        ### 代码编写结束 ###
    ], name = "Simple"
)
model_s.compile(
    ### 在此处编写代码 ###
    loss=None,
    optimizer=None,
    ### 在此处编写代码 ###
)


In [ ]:
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)

# 单元测试开始
model_s.fit(
    X_train,y_train,
    epochs=1000
)
# 单元测试结束

In [ ]:
# 单元测试开始
model_s.summary()

model_s_test(model_s, classes, X_train.shape[1])
# 单元测试结束

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
摘要应与下面一致（层实例名称可能递增）

```
Model: "Simple"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 6)                 18        
_________________________________________________________________
L2 (Dense)                   (None, 6)                 42        
=================================================================
Total params: 60
Trainable params: 60
Non-trainable params: 0
_________________________________________________________________
```

  <details>
  <summary><font size="3" color="darkgreen"><b>点击查看更多提示</b></font></summary>
  

```python
tf.random.set_seed(1234)
model_s = Sequential(
    [
        Dense(6, activation = 'relu', name="L1"),            # @REPLACE
        Dense(classes, activation = 'linear', name="L2")     # @REPLACE
    ], name = "Simple"
)
model_s.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),     # @REPLACE
    optimizer=tf.keras.optimizers.Adam(0.01),     # @REPLACE
)

model_s.fit(
    X_train,y_train,
    epochs=1000
)                                   
```
 

In [ ]:
#构建一个模型供绘图程序调用
model_predict_s = lambda Xl: np.argmax(tf.nn.softmax(model_s.predict(Xl)).numpy(),axis=1)
plt_nn(model_predict_s,X_train,y_train, classes, X_cv, y_cv, suptitle="Simple Model")

这个简单模型表现相当不错。让我们计算分类误差。

In [ ]:
training_cerr_simple = eval_cat_err(y_train, model_predict_s(X_train))
cv_cerr_simple = eval_cat_err(y_cv, model_predict_s(X_cv))
print(f"categorization error, training, simple model, {training_cerr_simple:0.3f}, complex model: {training_cerr_complex:0.3f}" )
print(f"categorization error, cv,       simple model, {cv_cerr_simple:0.3f}, complex model: {cv_cerr_complex:0.3f}" )

我们的简单模型在训练数据上的分类误差略高，但在交叉验证数据上比更复杂的模型表现更好。

<a name="6"></a>
## 6 - 正则化
与多项式回归的情形一样，可以应用正则化来缓和更复杂模型的影响。让我们在下面试一试。

<a name="ex05"></a>
### 练习 5

重建你的复杂模型，但这次加入正则化。
下面，构建一个三层模型：
* 120 个单元的 Dense 层，relu 激活函数，`kernel_regularizer=tf.keras.regularizers.l2(0.1)`
* 40 个单元的 Dense 层，relu 激活函数，`kernel_regularizer=tf.keras.regularizers.l2(0.1)`
* 6 个单元的 Dense 层，线性激活函数。  
编译时使用
* 配合 `SparseCategoricalCrossentropy` 的损失，记得使用 `from_logits=True`
* 学习率为 0.01 的 Adam 优化器。

In [ ]:
# UNQ_C5
# 计分单元：model_r

tf.random.set_seed(1234)
model_r = Sequential(
    [
        ### 在此处编写代码 ###
        
        ### 在此处编写代码 ###
    ], name= None
)
model_r.compile(
    ### 在此处编写代码 ###
    loss=None,
    optimizer=None,
    ### 在此处编写代码 ###
)


In [ ]:
# 单元测试开始
model_r.fit(
    X_train, y_train,
    epochs=1000
)
# 单元测试结束

In [ ]:
# 单元测试开始
model_r.summary()

model_r_test(model_r, classes, X_train.shape[1]) 
# 单元测试结束

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    
摘要应与下面一致（层实例名称可能递增）

```
Model: "ComplexRegularized"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
=================================================================
L1 (Dense)                   (None, 120)               360       
_________________________________________________________________
L2 (Dense)                   (None, 40)                4840      
_________________________________________________________________
L3 (Dense)                   (None, 6)                 246       
=================================================================
Total params: 5,446
Trainable params: 5,446
Non-trainable params: 0
_________________________________________________________________
```

  <details>
  <summary><font size="3" color="darkgreen"><b>点击查看更多提示</b></font></summary>
  

```python
tf.random.set_seed(1234)
model_r = Sequential(
    [
        Dense(120, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(0.1), name="L1"), 
        Dense(40, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(0.1), name="L2"),  
        Dense(classes, activation = 'linear', name="L3")  
    ], name="ComplexRegularized"
)
model_r.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    optimizer=tf.keras.optimizers.Adam(0.01),                             
)

model_r.fit(
    X_train,y_train,
    epochs=1000
)                                   
```
 

In [ ]:
#构建一个模型供绘图程序调用
model_predict_r = lambda Xl: np.argmax(tf.nn.softmax(model_r.predict(Xl)).numpy(),axis=1)
 
plt_nn(model_predict_r, X_train,y_train, classes, X_cv, y_cv, suptitle="Regularized")

结果看起来与“理想”模型非常相似。让我们检查分类误差。

In [ ]:
training_cerr_reg = eval_cat_err(y_train, model_predict_r(X_train))
cv_cerr_reg = eval_cat_err(y_cv, model_predict_r(X_cv))
test_cerr_reg = eval_cat_err(y_test, model_predict_r(X_test))
print(f"categorization error, training, regularized: {training_cerr_reg:0.3f}, simple model, {training_cerr_simple:0.3f}, complex model: {training_cerr_complex:0.3f}" )
print(f"categorization error, cv,       regularized: {cv_cerr_reg:0.3f}, simple model, {cv_cerr_simple:0.3f}, complex model: {cv_cerr_complex:0.3f}" )

简单模型在训练集上比正则化模型略好，但在交叉验证集上更差。

<a name="7"></a>
## 7 - 迭代寻找最优正则化取值
正如你在线性回归中所做的，可以尝试许多正则化取值。这段代码需要运行几分钟。如果你有时间，可以运行它并检查结果。如果没有，你已经完成了本作业的计分部分！

In [ ]:
tf.random.set_seed(1234)
lambdas = [0.0, 0.001, 0.01, 0.05, 0.1, 0.2, 0.3]
models=[None] * len(lambdas)
for i in range(len(lambdas)):
    lambda_ = lambdas[i]
    models[i] =  Sequential(
        [
            Dense(120, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)),
            Dense(40, activation = 'relu', kernel_regularizer=tf.keras.regularizers.l2(lambda_)),
            Dense(classes, activation = 'linear')
        ]
    )
    models[i].compile(
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        optimizer=tf.keras.optimizers.Adam(0.01),
    )

    models[i].fit(
        X_train,y_train,
        epochs=1000
    )
    print(f"Finished lambda = {lambda_}")


In [ ]:
plot_iterate(lambdas, models, X_train, y_train, X_cv, y_cv)

随着正则化增强，模型在训练数据集和交叉验证数据集上的表现趋于一致。对这个数据集和模型而言，lambda > 0.01 似乎是一个合理的选择。

<a name="7.1"></a>
### 7.1 测试
让我们在测试集上试用优化后的模型，并与“理想”表现做比较。  

In [ ]:
plt_compare(X_test,y_test, classes, model_predict_s, model_predict_r, centers)

我们的测试集很小，而且似乎有若干离群点，所以分类误差偏高。不过，我们优化后的模型表现与理想表现相当。

## 恭喜！
你已经熟悉了评估机器学习模型时可以采用的重要工具。具体来说：  
* 把数据划分为训练集和未训练集，使你能够区分欠拟合与过拟合
* 创建三个数据集——训练集、交叉验证集和测试集，使你能够
    * 用训练集训练参数 $W,B$
    * 用交叉验证集调整模型参数，如复杂度、正则化和样本数量
    * 用测试集评估你的“真实场景”表现
* 比较训练表现与交叉验证表现，可以洞察模型偏向过拟合（高方差）还是欠拟合（高偏差）